In [10]:
from google.colab import drive
drive.mount("/content/gdrive")

Mounted at /content/gdrive


In [11]:
%cd /content/gdrive/MyDrive/

/content/gdrive/MyDrive


In [12]:
!pip install ipython-autotime
%load_ext autotime

time: 312 µs (started: 2026-04-01 14:19:23 +00:00)


In [ ]:
!mv /datasets/101_ObjectCategories /datasets/caltech101
!rm -rf /datasets/caltech101/101_ObjectCategories

mv: cannot stat '/datasets/101_ObjectCategories': No such file or directory
time: 206 ms (started: 2026-03-31 11:18:37 +00:00)


# Model Selection

In [13]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pickle
from tqdm import tqdm, tqdm_notebook
import os
import random
import time
import math
import tensorflow
from tensorflow.keras.preprocessing import image
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.vgg19 import VGG19
from tensorflow.keras.applications.mobilenet import MobileNet
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Flatten, Dense, Dropout, GlobalAveragePooling2D


time: 16 ms (started: 2026-04-01 14:19:33 +00:00)


In [14]:
def model_picker(name):
    if (name == 'vgg16'):
        model = VGG16(weights='imagenet',
                      include_top=False,
                      input_shape=(224, 224, 3),
                      pooling='max')
    elif (name == 'vgg19'):
        model = VGG19(weights='imagenet',
                      include_top=False,
                      input_shape=(224, 224, 3),
                      pooling='max')
    elif (name == 'mobilenet'):
        model = MobileNet(weights='imagenet',
                          include_top=False,
                          input_shape=(224, 224, 3),
                          pooling='max',
                          depth_multiplier=1,
                          alpha=1)
    elif (name == 'inception'):
        model = InceptionV3(weights='imagenet',
                            include_top=False,
                            input_shape=(224, 224, 3),
                            pooling='max')
    elif (name == 'resnet'):
        model = ResNet50(weights='imagenet',
                         include_top=False,
                         input_shape=(224, 224, 3),
                        pooling='max')
    elif (name == 'xception'):
        model = Xception(weights='imagenet',
                         include_top=False,
                         input_shape=(224, 224, 3),
                         pooling='max')
    else:
        print("Specified model not available")
    return model

model_architecture = 'resnet'  # Resnet has largest fraction of non-zero values compared to other keras imagenet pretrained models: https://youtu.be/-5BAepEE9I8?t=524
model = model_picker(model_architecture)

time: 4.25 s (started: 2026-04-01 14:19:36 +00:00)


# Feature extraction (one by one)

In [15]:
def extract_features(img_path, model):
    input_shape = (224, 224, 3)
    img = image.load_img(img_path,
                         target_size=(input_shape[0], input_shape[1]))
    img_array = image.img_to_array(img)
    expanded_img_array = np.expand_dims(img_array, axis=0)   # create batch of 1
    preprocessed_img = preprocess_input(expanded_img_array)
    features = model.predict(preprocessed_img)
    flattened_features = features.flatten()

    return flattened_features

time: 1.07 ms (started: 2026-04-01 14:19:45 +00:00)


# Checking extracted features from same class {class1, image1}, {class1, image2}  are closer than features from different class (class1, image1), {class2, image1}
- Euclidean Distance
- Dot product
- Cosine similarity

In [16]:
import os

print(os.listdir())

['PixCollab', 'EdgeVision', 'certification-Exam---PingCAP-Certified-TiDB-Practitioner-Bhavna_08.pdf', 'BL.SC.P2CSE25027_Aadhaar.pdf', 'Poli_Bhavana_Resume (7).pdf', 'Poli_Bhavana_Resume (6).pdf', 'Poli_Bhavana_Resume (5).pdf', 'Poli_Bhavana_Resume (4).pdf', 'Poli_Bhavana_Resume (3).pdf', 'Poli_Bhavana_Resume (2).pdf', 'GooglePlayAcademy Certificate.pdf', 'Bhavana_Resume (2).pdf', 'pubmed-pharma-papers-report.pdf', 'Screen Recording 2025-07-28 at 5.02.25\u202fPM.mov', 'DELOITTE.pdf', 'Bhavana_Resume (1).pdf', 'Bhavana_Resume.pdf', 'CV_Bhavana_Poli_2025.pdf', 'Google analytics certificate.pdf', 'TATA AI GENERATIVE.pdf', 'Walmart forage.pdf', 'JP MORGAN CHASE & CO. INTERN.pdf', 'Accenture - forage.pdf', 'Oracle AI.pdf', 'Oracle Data Scientist.pdf', 'Bhavana_Poli_Resume (19).pdf', 'Bhavana_Poli_Resume (18).pdf', 'NEON.D12.YELL.DP1.00005.001.sensor_positions (1).20240906T020644Z.csv', 'NEON.D12.YELL.DP1.00005.001.sensor_positions.20240906T020644Z.csv', 'IR_Data.gsheet', 'NEON_Spatial_IR_Dat

In [20]:
bike1 = extract_features('caltech-101/101_ObjectCategories/Motorbikes/image_0001.jpg', model)

bike2 = extract_features('caltech-101/101_ObjectCategories/Motorbikes/image_0002.jpg', model)

plane1 = extract_features('caltech-101/101_ObjectCategories/airplanes/image_0001.jpg', model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
time: 6.34 s (started: 2026-04-01 14:21:20 +00:00)


In [21]:
def compare_metrics(c1ass1image1, class1image2, class2image1):
  def cosine_similarity(list_1, list_2):
    cos_sim = dot(list_1, list_2) / (norm(list_1) * norm(list_2))
    return cos_sim

  print('Euclidean: ', np.linalg.norm(c1ass1image1-class1image2), np.linalg.norm(c1ass1image1-class2image1))  # first pair of images should have smaller distance
  print('Dot product:', c1ass1image1@class1image2, c1ass1image1@class2image1) # first pair of images should have larger dot product
  print('Cosine Similarity: ', cosine_similarity(c1ass1image1,class1image2), cosine_similarity(c1ass1image1,class2image1)) # first pair of images should have larger cosine similarity

compare_metrics(bike1, bike2, plane1)

Euclidean:  180.74825 295.42456
Dot product: 92433.37 58649.715
Cosine Similarity:  0.8502075 0.5736602
time: 1.75 ms (started: 2026-04-01 14:21:30 +00:00)


# Getting all filenames
- Used for visualization later

In [22]:
!du -a caltech-101

Streaming output truncated to the last 5000 lines.
23	caltech-101/101_ObjectCategories/beaver/image_0044.jpg
24	caltech-101/101_ObjectCategories/beaver/image_0015.jpg
10	caltech-101/101_ObjectCategories/beaver/image_0003.jpg
717	caltech-101/101_ObjectCategories/beaver
23	caltech-101/101_ObjectCategories/wheelchair/image_0052.jpg
18	caltech-101/101_ObjectCategories/wheelchair/image_0014.jpg
21	caltech-101/101_ObjectCategories/wheelchair/image_0040.jpg
17	caltech-101/101_ObjectCategories/wheelchair/image_0035.jpg
20	caltech-101/101_ObjectCategories/wheelchair/image_0019.jpg
19	caltech-101/101_ObjectCategories/wheelchair/image_0018.jpg
14	caltech-101/101_ObjectCategories/wheelchair/image_0044.jpg
23	caltech-101/101_ObjectCategories/wheelchair/image_0038.jpg
14	caltech-101/101_ObjectCategories/wheelchair/image_0024.jpg
19	caltech-101/101_ObjectCategories/wheelchair/image_0048.jpg
20	caltech-101/101_ObjectCategories/wheelchair/image_0004.jpg
19	caltech-101/101_ObjectCategories/wheelchair/im

In [23]:
!du -a caltech-101 | cut -d/ -f3 | sort | uniq -c | sort -nr

    801 Airplanes_Side_2
    801 airplanes
    799 Motorbikes_16
    799 Motorbikes
    480 watch
    469 BACKGROUND_Google
    436 Faces_easy
    436 Faces_3
    436 Faces_2
    436 Faces
    402 Leopards
    258 bonsai
    248 car_side
    230 ketch
    216 chandelier
    202 hawksbill
    200 grand_piano
    198 brain
    184 butterfly
    178 helicopter
    176 menorah
    174 trilobite
    174 starfish
    174 kangaroo
    172 sunflower
    172 ewer
    172 buddha
    170 scorpion
    166 revolver
    164 laptop
    162 ibis
    158 llama
    154 minaret
    152 umbrella
    152 electric_guitar
    148 crab
    142 crayfish
    140 cougar_face
    138 dragonfly
    136 flamingo
    136 ferry
    136 dalmatian
    134 lotus
    132 dolphin
    130 stop_sign
    130 soccer_ball
    130 joshua_tree
    130 euphonium
    130 elephant
    128 schooner
    126 chair
    124 lamp
    122 yin_yang
    120 wheelchair
    120 stegosaurus
    120 rhino
    120 cellphone
    116 sea_horse
   

In [24]:
extensions = ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']

def get_file_list(root_dir):
    file_list = []
    for root, directories, filenames in os.walk(root_dir):
        for filename in filenames:
            if any(ext in filename for ext in extensions):
                filepath = os.path.join(root, filename)
                if os.path.exists(filepath):
                  file_list.append(filepath)
                else:
                  print(filepath)
    return file_list

root_dir = 'caltech-101/101_ObjectCategories'
filenames = sorted(get_file_list(root_dir))
print(len(filenames))

9144
time: 1.24 s (started: 2026-04-01 14:22:02 +00:00)


# Feature extraction (tensorflow batches)

In [25]:
datagen = tensorflow.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)

generator = datagen.flow_from_directory(root_dir,
                                        target_size=(224, 224),
                                        class_mode=None,
                                        shuffle=False,
                                        batch_size=64)

feature_list = model.predict(generator, verbose=1)

Found 9144 images belonging to 102 classes.
143/143 ━━━━━━━━━━━━━━━━━━━━ 65s 394ms/step
time: 1min 7s (started: 2026-04-01 14:22:06 +00:00)


In [26]:
feature_list.shape

(9144, 2048)

time: 1.82 ms (started: 2026-04-01 14:23:20 +00:00)


In [ ]:
!mkdir -p /features
pickle.dump(generator.classes, open('/features/class_ids-caltech101.pickle','wb'))
pickle.dump(filenames, open('/features/filenames-caltech101.pickle', 'wb'))
pickle.dump(feature_list,open('/features/features-caltech101-' + model_architecture + '.pickle', 'wb'))

time: 187 ms (started: 2026-03-31 12:58:05 +00:00)


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create features folder in Drive
import os
FEATURE_PATH = "/content/drive/MyDrive/features"
os.makedirs(FEATURE_PATH, exist_ok=True)

# Save pickle files
import pickle

pickle.dump(generator.classes, open(FEATURE_PATH + '/class_ids-caltech101.pickle','wb'))
pickle.dump(filenames, open(FEATURE_PATH + '/filenames-caltech101.pickle', 'wb'))
pickle.dump(feature_list, open(FEATURE_PATH + '/features-caltech101-' + model_architecture + '.pickle', 'wb'))

Mounted at /content/drive
time: 6.74 s (started: 2026-03-31 12:59:11 +00:00)


# Training Model from scratch (without final dense layers)

- Aim to learn better embeddings specific to this data
- No dense layers are added except final one to classify so the model can put all weight on convolutional layers instead of dense layers that will be truncated anyway after training for feature extraction  

In [11]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive
time: 5.01 s (started: 2026-04-01 13:15:14 +00:00)


In [12]:
root_dir = "/content/drive/MyDrive/caltech-101/101_ObjectCategories"

time: 313 µs (started: 2026-04-01 13:15:23 +00:00)


In [13]:
train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                   rotation_range=20,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   zoom_range=0.2)

train_generator = train_datagen.flow_from_directory(root_dir,
                                                    target_size=(224, 224),
                                                    shuffle=True,
                                                    seed=12345,
                                                    class_mode='categorical')

NUM_CLASSES = 102

model = ResNet50(weights='imagenet', include_top=False,input_shape = (224,224,3))
input = Input(shape=(224, 224, 3))
x = model(input)
x = GlobalAveragePooling2D()(x)
# No extra dense or dropout layers so heavy lifting for classification accuracy rests on convolution layers
x = Dense(NUM_CLASSES, activation='softmax')(x)
model_similarity_optimized = Model(inputs=input, outputs=x)

Found 9144 images belonging to 102 classes.
time: 12.7 s (started: 2026-04-01 13:15:25 +00:00)


In [ ]:
model_similarity_optimized.compile(loss='categorical_crossentropy',
              optimizer=tensorflow.keras.optimizers.Adam(0.001),
              metrics=['acc'])
model_similarity_optimized.fit(train_generator,
                               batch_size=64,
                               epochs=10)

Epoch 1/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 272s 721ms/step - acc: 0.4364 - loss: 2.5975
Epoch 2/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 176s 614ms/step - acc: 0.6963 - loss: 1.1612
Epoch 3/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 176s 614ms/step - acc: 0.7877 - loss: 0.7873
Epoch 4/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 175s 611ms/step - acc: 0.8423 - loss: 0.5574
Epoch 5/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 175s 612ms/step - acc: 0.8710 - loss: 0.4562
Epoch 6/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 174s 607ms/step - acc: 0.8864 - loss: 0.3818
Epoch 7/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 178s 620ms/step - acc: 0.8999 - loss: 0.3458
Epoch 8/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 176s 614ms/step - acc: 0.9164 - loss: 0.2849
Epoch 9/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 175s 612ms/step - acc: 0.9231 - loss: 0.2556
Epoch 10/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 178s 620ms/step - acc: 0.9253 - loss: 0.2490


time: 30min 55s (started: 2026-03-31 13:29:57 +00:00)


In [ ]:
!mkdir -p /models
model_similarity_optimized.save('/models/model-scratch.h5')

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


time: 1.19 s (started: 2026-03-31 14:21:45 +00:00)


In [ ]:
model_similarity_optimized.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 102)            │       208,998 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,796,710 (90.78 MB)

 Trainable params: 23,743,590 (90.57 MB)

 Non-trainable params: 53,120 (207.50 KB)

time: 27.3 ms (started: 2026-03-31 16:21:38 +00:00)


In [ ]:
model = Model(model_similarity_optimized.input, model_similarity_optimized.layers[-2].output)
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 23,534,592 (89.78 MB)

 Non-trainable params: 53,120 (207.50 KB)

time: 29.9 ms (started: 2026-03-31 16:21:40 +00:00)


In [14]:
import os
print(os.listdir('/content/drive'))

['MyDrive', '.shortcut-targets-by-id', '.Trash-0', '.Encrypted']
time: 1.24 ms (started: 2026-04-01 13:16:12 +00:00)


In [15]:
import os
print(os.listdir('/content/drive/MyDrive'))

['OmniOrderMS', 'PixCollab', 'EdgeVision', 'certification-Exam---PingCAP-Certified-TiDB-Practitioner-Bhavna_08.pdf', 'BL.SC.P2CSE25027_Aadhaar.pdf', 'Poli_Bhavana_Resume (7).pdf', 'Poli_Bhavana_Resume (6).pdf', 'Poli_Bhavana_Resume (5).pdf', 'Poli_Bhavana_Resume (4).pdf', 'Poli_Bhavana_Resume (3).pdf', 'Poli_Bhavana_Resume (2).pdf', 'GooglePlayAcademy Certificate.pdf', 'Bhavana_Resume (2).pdf', 'pubmed-pharma-papers-report.pdf', 'Screen Recording 2025-07-28 at 5.02.25\u202fPM.mov', 'DELOITTE.pdf', 'Bhavana_Resume (1).pdf', 'Bhavana_Resume.pdf', 'CV_Bhavana_Poli_2025.pdf', 'Google analytics certificate.pdf', 'TATA AI GENERATIVE.pdf', 'Walmart forage.pdf', 'JP MORGAN CHASE & CO. INTERN.pdf', 'Accenture - forage.pdf', 'Oracle AI.pdf', 'Oracle Data Scientist.pdf', 'Bhavana_Poli_Resume (19).pdf', 'Bhavana_Poli_Resume (18).pdf', 'NEON.D12.YELL.DP1.00005.001.sensor_positions (1).20240906T020644Z.csv', 'NEON.D12.YELL.DP1.00005.001.sensor_positions.20240906T020644Z.csv', 'IR_Data.gsheet', 'NEON

In [17]:
!rm -rf /content/drive

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
^C
time: 2.05 s (started: 2026-04-01 13:20:18 +00:00)


In [18]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
time: 3.4 s (started: 2026-04-01 13:20:23 +00:00)


In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
time: 3.4 s (started: 2026-04-01 13:20:36 +00:00)


In [20]:
import os
print(os.listdir('/content/drive/MyDrive'))

['PixCollab', 'EdgeVision', 'certification-Exam---PingCAP-Certified-TiDB-Practitioner-Bhavna_08.pdf', 'BL.SC.P2CSE25027_Aadhaar.pdf', 'Poli_Bhavana_Resume (7).pdf', 'Poli_Bhavana_Resume (6).pdf', 'Poli_Bhavana_Resume (5).pdf', 'Poli_Bhavana_Resume (4).pdf', 'Poli_Bhavana_Resume (3).pdf', 'Poli_Bhavana_Resume (2).pdf', 'GooglePlayAcademy Certificate.pdf', 'Bhavana_Resume (2).pdf', 'pubmed-pharma-papers-report.pdf', 'Screen Recording 2025-07-28 at 5.02.25\u202fPM.mov', 'DELOITTE.pdf', 'Bhavana_Resume (1).pdf', 'Bhavana_Resume.pdf', 'CV_Bhavana_Poli_2025.pdf', 'Google analytics certificate.pdf', 'TATA AI GENERATIVE.pdf', 'Walmart forage.pdf', 'JP MORGAN CHASE & CO. INTERN.pdf', 'Accenture - forage.pdf', 'Oracle AI.pdf', 'Oracle Data Scientist.pdf', 'Bhavana_Poli_Resume (19).pdf', 'Bhavana_Poli_Resume (18).pdf', 'NEON.D12.YELL.DP1.00005.001.sensor_positions (1).20240906T020644Z.csv', 'NEON.D12.YELL.DP1.00005.001.sensor_positions.20240906T020644Z.csv', 'IR_Data.gsheet', 'NEON_Spatial_IR_Dat

In [21]:
import os
print(os.listdir('/content/drive/MyDrive/caltech-101'))

['101_ObjectCategories.tar.gz', 'show_annotation.m', 'Annotations.tar', '.DS_Store', 'Annotations', '101_ObjectCategories']
time: 3.58 ms (started: 2026-04-01 13:20:54 +00:00)


In [22]:
root_dir = "/content/drive/MyDrive/caltech-101/101_ObjectCategories"

time: 317 µs (started: 2026-04-01 13:20:56 +00:00)


In [23]:
import os
print(os.listdir('/content/drive/MyDrive/features'))

['class_ids-caltech101.pickle', 'filenames-caltech101.pickle', 'features-caltech101-resnet.pickle', 'features-caltech101-resnet-scratch.pickle']
time: 1.22 ms (started: 2026-04-01 13:20:57 +00:00)


In [ ]:
datagen = tensorflow.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=preprocess_input
)

generator_refined = datagen.flow_from_directory(
    root_dir,
    target_size=(224, 224),
    class_mode=None,
    shuffle=False,
    batch_size=64
)

# ✅ IMPORTANT FIX HERE
feature_list_scratch = model.predict(generator_refined, verbose=1)

Found 9144 images belonging to 102 classes.
143/143 ━━━━━━━━━━━━━━━━━━━━ 3297s 23s/step


In [1]:
import os
print(os.listdir('/content/drive/MyDrive'))

['PixCollab', 'EdgeVision', 'certification-Exam---PingCAP-Certified-TiDB-Practitioner-Bhavna_08.pdf', 'BL.SC.P2CSE25027_Aadhaar.pdf', 'Poli_Bhavana_Resume (7).pdf', 'Poli_Bhavana_Resume (6).pdf', 'Poli_Bhavana_Resume (5).pdf', 'Poli_Bhavana_Resume (4).pdf', 'Poli_Bhavana_Resume (3).pdf', 'Poli_Bhavana_Resume (2).pdf', 'GooglePlayAcademy Certificate.pdf', 'Bhavana_Resume (2).pdf', 'pubmed-pharma-papers-report.pdf', 'Screen Recording 2025-07-28 at 5.02.25\u202fPM.mov', 'DELOITTE.pdf', 'Bhavana_Resume (1).pdf', 'Bhavana_Resume.pdf', 'CV_Bhavana_Poli_2025.pdf', 'Google analytics certificate.pdf', 'TATA AI GENERATIVE.pdf', 'Walmart forage.pdf', 'JP MORGAN CHASE & CO. INTERN.pdf', 'Accenture - forage.pdf', 'Oracle AI.pdf', 'Oracle Data Scientist.pdf', 'Bhavana_Poli_Resume (19).pdf', 'Bhavana_Poli_Resume (18).pdf', 'NEON.D12.YELL.DP1.00005.001.sensor_positions (1).20240906T020644Z.csv', 'NEON.D12.YELL.DP1.00005.001.sensor_positions.20240906T020644Z.csv', 'IR_Data.gsheet', 'NEON_Spatial_IR_Dat

In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

In [ ]:
datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

generator_refined = datagen.flow_from_directory(
    root_dir,
    target_size=(224, 224),
    class_mode=None,
    shuffle=False,
    batch_size=64
)

Found 9144 images belonging to 102 classes.
time: 345 ms (started: 2026-04-01 05:25:12 +00:00)


In [ ]:
pickle.dump(feature_list_scratch,open('/features/features-caltech101-' + model_architecture + 'scratch' + '.pickle', 'wb'))

FileNotFoundError: [Errno 2] No such file or directory: '/features/features-caltech101-resnetscratch.pickle'

time: 5.09 ms (started: 2026-03-31 17:17:22 +00:00)


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/features'))

['class_ids-caltech101.pickle', 'filenames-caltech101.pickle', 'features-caltech101-resnet.pickle', 'features-caltech101-resnet-scratch.pickle']
time: 2.91 ms (started: 2026-04-01 05:27:11 +00:00)


In [ ]:
import os
import pickle

FEATURE_PATH = "/content/drive/MyDrive/features"
os.makedirs(FEATURE_PATH, exist_ok=True)

pickle.dump(
    feature_list_scratch,
    open(FEATURE_PATH + '/features-caltech101-' + model_architecture + '-scratch.pickle', 'wb')
)

NameError: name 'feature_list_scratch' is not defined

time: 4.31 ms (started: 2026-04-01 05:28:26 +00:00)


In [ ]:
import os

FEATURE_PATH = "/content/drive/MyDrive/features"
os.makedirs(FEATURE_PATH, exist_ok=True)

time: 3.11 ms (started: 2026-03-31 17:17:39 +00:00)


In [ ]:
import pickle

pickle.dump(
    feature_list_scratch,
    open(FEATURE_PATH + '/features-caltech101-' + model_architecture + '-scratch.pickle', 'wb')
)

time: 224 ms (started: 2026-03-31 17:17:41 +00:00)


In [ ]:
import os
print(os.listdir('/content'))

['.config', 'gdrive', 'drive', 'sample_data']
time: 1.21 ms (started: 2026-03-31 17:17:43 +00:00)


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['OmniOrderMS', 'PixCollab', 'EdgeVision', 'certification-Exam---PingCAP-Certified-TiDB-Practitioner-Bhavna_08.pdf', 'BL.SC.P2CSE25027_Aadhaar.pdf', 'Poli_Bhavana_Resume (7).pdf', 'Poli_Bhavana_Resume (6).pdf', 'Poli_Bhavana_Resume (5).pdf', 'Poli_Bhavana_Resume (4).pdf', 'Poli_Bhavana_Resume (3).pdf', 'Poli_Bhavana_Resume (2).pdf', 'GooglePlayAcademy Certificate.pdf', 'Bhavana_Resume (2).pdf', 'pubmed-pharma-papers-report.pdf', 'Screen Recording 2025-07-28 at 5.02.25\u202fPM.mov', 'DELOITTE.pdf', 'Bhavana_Resume (1).pdf', 'Bhavana_Resume.pdf', 'CV_Bhavana_Poli_2025.pdf', 'Google analytics certificate.pdf', 'TATA AI GENERATIVE.pdf', 'Walmart forage.pdf', 'JP MORGAN CHASE & CO. INTERN.pdf', 'Accenture - forage.pdf', 'Oracle AI.pdf', 'Oracle Data Scientist.pdf', 'Bhavana_Poli_Resume (19).pdf', 'Bhavana_Poli_Resume (18).pdf', 'NEON.D12.YELL.DP1.00005.001.sensor_positions (1).20240906T020644Z.csv', 'NEON.D12.YELL.DP1.00005.001.sensor_positions.20240906T020644Z.csv', 'IR_Data.gsheet', 'NEON

In [ ]:
root_dir = "/content/drive/MyDrive/caltech-101/101_ObjectCategories"

time: 560 µs (started: 2026-03-31 17:17:47 +00:00)


In [ ]:
bike1 = extract_features(
    '/content/drive/MyDrive/caltech-101/101_ObjectCategories/Motorbikes/image_0001.jpg',
    model
)

bike2 = extract_features(
    '/content/drive/MyDrive/caltech-101/101_ObjectCategories/Motorbikes/image_0002.jpg',
    model
)

plane1 = extract_features(
    '/content/drive/MyDrive/caltech-101/101_ObjectCategories/airplanes/image_0001.jpg',
    model
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 230ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 225ms/step
time: 4.94 s (started: 2026-03-31 17:17:48 +00:00)


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def compare_metrics(vec1, vec2, vec3):

    # Euclidean distance
    eucl_12 = np.linalg.norm(vec1 - vec2)
    eucl_13 = np.linalg.norm(vec1 - vec3)

    # Dot product
    dot_12 = np.dot(vec1, vec2)
    dot_13 = np.dot(vec1, vec3)

    # Cosine similarity
    cos_12 = cosine_similarity([vec1], [vec2])[0][0]
    cos_13 = cosine_similarity([vec1], [vec3])[0][0]

    print("Euclidean: ", eucl_12, eucl_13)
    print("Dot product:", dot_12, dot_13)
    print("Cosine Similarity: ", cos_12, cos_13)

time: 1.28 ms (started: 2026-03-31 17:23:08 +00:00)


In [ ]:
compare_metrics(bike1, bike2, plane1)

Euclidean:  20.424095 36.797855
Dot product: 1254.4863 554.8417
Cosine Similarity:  0.8590414 0.4534201
time: 1.56 ms (started: 2026-03-31 17:24:34 +00:00)


## Interpretations
- it's strange that the dot product between bike1, plane1 is higher than bike1, bike2 (these two are visually almost the same)
- Euclidean distances both shrunk significantly compared to directly using resnet
- Cosine Similarity both increased
- Aiming to get small distance/big similarity between first pair and big gap in metrics between pair1 and pair2


In [ ]:
cougarbody1 = extract_features('/content/drive/MyDrive/caltech-101/101_ObjectCategories/cougar_body/image_0002.jpg', model)
cougarbody2 = extract_features('/content/drive/MyDrive/caltech-101/101_ObjectCategories/cougar_body/image_0001.jpg', model)
cougarface1 = extract_features('/content/drive/MyDrive/caltech-101/101_ObjectCategories/cougar_face/image_0001.jpg', model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 226ms/step
time: 4.03 s (started: 2026-03-31 17:25:08 +00:00)


In [ ]:
compare_metrics(cougarbody1,cougarbody2,cougarface1)

Euclidean:  531.3156 678.7931
Dot product: 182823.19 80120.8
Cosine Similarity:  0.5663034 0.25828618
time: 3.56 ms (started: 2026-03-31 17:25:14 +00:00)


In [5]:
%cd /content

/content


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
time: 3.42 s (started: 2026-03-31 17:26:00 +00:00)


In [6]:
!mkdir -p /content/drive/MyDrive/ahrefs

In [ ]:
!cp -r /features /content/drive/MyDrive/ahrefs/features
!cp -r /models /content/drive/MyDrive/ahrefs/models


cp: cannot stat '/features': No such file or directory
cp: cannot stat '/models': No such file or directory
time: 213 ms (started: 2026-03-31 17:26:08 +00:00)


In [ ]:
model.save('/content/drive/MyDrive/ahrefs/models/resnet_model.h5')

time: 602 ms (started: 2026-03-31 14:57:47 +00:00)


# Fine tuning model
- Because didn't train from scratch long enough to give better accuracy against true labels compared to pre-trained model

In [27]:
def model_maker():
    base_model = ResNet50(include_top=False,
                           input_shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    for layer in base_model.layers[:]:
        layer.trainable = False
    input = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))
    custom_model = base_model(input)
    custom_model = GlobalAveragePooling2D()(custom_model)
    custom_model = Dense(64, activation='relu')(custom_model)
    custom_model = Dropout(0.5)(custom_model)
    predictions = Dense(NUM_CLASSES, activation='softmax')(custom_model)
    return Model(inputs=input, outputs=predictions)

time: 807 µs (started: 2026-04-01 14:23:40 +00:00)


In [28]:
NUM_CLASSES = 102
IMG_WIDTH, IMG_HEIGHT = 224, 224

train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input,
                                   rotation_range=20,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   zoom_range=0.2)

train_generator = train_datagen.flow_from_directory(root_dir,
                                                    target_size=(IMG_WIDTH,
                                                                 IMG_HEIGHT),
                                                    shuffle=True,
                                                    seed=12345,
                                                    class_mode='categorical')

model_finetuned = model_maker()
model_finetuned.compile(loss='categorical_crossentropy',
              optimizer=tensorflow.keras.optimizers.Adam(0.001),
              metrics=['acc'])
model_finetuned.fit(train_generator,
                    epochs=10)

Found 9144 images belonging to 102 classes.
Epoch 1/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 165s 527ms/step - acc: 0.4189 - loss: 2.6382
Epoch 2/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 146s 511ms/step - acc: 0.6132 - loss: 1.4900
Epoch 3/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 144s 503ms/step - acc: 0.6829 - loss: 1.1670
Epoch 4/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 144s 504ms/step - acc: 0.7205 - loss: 1.0105
Epoch 5/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 146s 511ms/step - acc: 0.7425 - loss: 0.9037
Epoch 6/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 145s 506ms/step - acc: 0.7539 - loss: 0.8499
Epoch 7/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 145s 507ms/step - acc: 0.7692 - loss: 0.7874
Epoch 8/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 145s 508ms/step - acc: 0.7776 - loss: 0.7523
Epoch 9/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 147s 513ms/step - acc: 0.7818 - loss: 0.7391
Epoch 10/10
286/286 ━━━━━━━━━━━━━━━━━━━━ 146s 510ms/step - acc: 0.7881 - loss: 0.7147


time: 24min 36s (started: 2026-04-01 14:23:42 +00:00)


In [29]:
!mkdir -p /models
model_finetuned.save('/models/model-resnet-finetuned.h5')

time: 538 ms (started: 2026-04-01 14:48:24 +00:00)


In [31]:
!fusermount -u /content/drive

time: 105 ms (started: 2026-04-01 14:52:08 +00:00)


In [32]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
time: 4.9 s (started: 2026-04-01 14:52:19 +00:00)


In [33]:
import os

MODEL_PATH = "/content/drive/MyDrive/ahrefs/models"
os.makedirs(MODEL_PATH, exist_ok=True)

model_finetuned.save(MODEL_PATH + '/model-resnet-finetuned.keras')

time: 7.47 s (started: 2026-04-01 14:52:31 +00:00)


In [34]:
model_finetuned.save('/models/model-resnet-finetuned.keras')

time: 1.02 s (started: 2026-04-01 14:52:59 +00:00)


In [35]:
import os

MODEL_PATH = "/content/drive/MyDrive/ahrefs/models"
os.makedirs(MODEL_PATH, exist_ok=True)

model_finetuned.save(MODEL_PATH + '/model-resnet-finetuned.keras')

time: 858 ms (started: 2026-04-01 14:53:30 +00:00)


In [36]:
import os
print(os.listdir("/content/drive/MyDrive/ahrefs/models"))

['model-scratch.h5', 'resnet_model.h5', 'model-resnet-finetuned.keras']
time: 1.5 ms (started: 2026-04-01 14:53:38 +00:00)


In [37]:
model_finetuned_extractor = Model(model_finetuned.input, model_finetuned.layers[-4].output)
model_finetuned_extractor.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

time: 20.5 ms (started: 2026-04-01 14:54:22 +00:00)


In [39]:
datagen = tensorflow.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)

root_dir='/content/drive/MyDrive/caltech-101/101_ObjectCategories'

generator_refined = datagen.flow_from_directory(root_dir,
                                        target_size=(224, 224),
                                        class_mode=None,
                                        shuffle=False,
                                        batch_size=64)

feature_list_finetuned = model_finetuned_extractor.predict(generator_refined, verbose=1)
feature_list_finetuned.shape # ensure 2048 dimensions

Found 9144 images belonging to 102 classes.
143/143 ━━━━━━━━━━━━━━━━━━━━ 58s 377ms/step


(9144, 2048)

time: 1min (started: 2026-04-01 14:56:29 +00:00)


In [41]:
%cd /content

/content
time: 2.1 ms (started: 2026-04-01 14:59:53 +00:00)


In [42]:
import os
import pickle

# create folder
os.makedirs('/content/features', exist_ok=True)

# save file
pickle.dump(
    feature_list_finetuned,
    open('/content/features/features-caltech101-resnet-finetuned.pickle', 'wb')
)

time: 84.6 ms (started: 2026-04-01 15:00:05 +00:00)


In [46]:
import os
import pickle

FEATURE_PATH = "/content/drive/MyDrive/ahrefs/features"
os.makedirs(FEATURE_PATH, exist_ok=True)

pickle.dump(
    feature_list_finetuned,
    open(FEATURE_PATH + '/features-caltech101-resnet-finetuned.pickle', 'wb')
)

time: 168 ms (started: 2026-04-01 15:01:43 +00:00)


In [40]:
!mkdir -p /features
pickle.dump(feature_list_finetuned,open('/features/features-caltech101-resnet-finetuned.pickle', 'wb'))

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
time: 190 ms (started: 2026-04-01 14:57:33 +00:00)


In [43]:
!cp /features/features-caltech101-resnet-finetuned.pickle /content/gdrive/MyDrive/ahrefs/features/features-caltech101-resnet-finetuned.pickle

cp: failed to access '/content/gdrive/MyDrive/ahrefs/features/features-caltech101-resnet-finetuned.pickle': Transport endpoint is not connected
time: 106 ms (started: 2026-04-01 15:00:15 +00:00)


In [44]:
!cp /content/features/features-caltech101-resnet-finetuned.pickle \
/content/drive/MyDrive/ahrefs/features/

time: 505 ms (started: 2026-04-01 15:00:56 +00:00)


In [48]:
bike1 = extract_features(
    '/content/drive/MyDrive/caltech-101/101_ObjectCategories/Motorbikes/image_0001.jpg',
    model_finetuned_extractor
)

bike2 = extract_features(
    '/content/drive/MyDrive/caltech-101/101_ObjectCategories/Motorbikes/image_0002.jpg',
    model_finetuned_extractor
)

plane1 = extract_features(
    '/content/drive/MyDrive/caltech-101/101_ObjectCategories/airplanes/image_0001.jpg',
    model_finetuned_extractor
)

def compare_metrics(c1ass1image1, class1image2, class2image1):
  def cosine_similarity(list_1, list_2):
    cos_sim = dot(list_1, list_2) / (norm(list_1) * norm(list_2))
    return cos_sim

  print('Euclidean: ', np.linalg.norm(c1ass1image1-class1image2), np.linalg.norm(c1ass1image1-class2image1))  # first pair of images should have smaller distance
  print('Dot product:', c1ass1image1@class1image2, c1ass1image1@class2image1) # first pair of images should have larger dot product
  print('Cosine Similarity: ', cosine_similarity(c1ass1image1,class1image2), cosine_similarity(c1ass1image1,class2image1)) # first pair of images should have larger cosine similarity

compare_metrics(bike1, bike2, plane1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
Euclidean:  20.424093 36.797867
Dot product: 1254.487 554.84204
Cosine Similarity:  0.8590416 0.45342016
time: 3.87 s (started: 2026-04-01 15:02:43 +00:00)


# VOC2012
- Harder image collection where multiple labels exist per image

In [1]:
!wget -P /datasets "http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar"

--2026-04-01 15:06:57--  http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
Resolving host.robots.ox.ac.uk (host.robots.ox.ac.uk)... 129.67.94.50
Connecting to host.robots.ox.ac.uk (host.robots.ox.ac.uk)|129.67.94.50|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar [following]
--2026-04-01 15:06:57--  https://www.robots.ox.ac.uk/~vgg/projects/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
Resolving www.robots.ox.ac.uk (www.robots.ox.ac.uk)... 129.67.94.2
Connecting to www.robots.ox.ac.uk (www.robots.ox.ac.uk)|129.67.94.2|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar [following]
--2026-04-01 15:06:59--  https://thor.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar
Resolving thor.robots.ox.ac.uk (thor.ro

In [2]:
!tar -xvf /datasets/VOCtrainval_11-May-2012.tar --directory /datasets

Streaming output truncated to the last 5000 lines.
VOCdevkit/VOC2012/SegmentationClass/2008_001874.png
VOCdevkit/VOC2012/SegmentationClass/2008_001876.png
VOCdevkit/VOC2012/SegmentationClass/2008_001882.png
VOCdevkit/VOC2012/SegmentationClass/2008_001885.png
VOCdevkit/VOC2012/SegmentationClass/2008_001895.png
VOCdevkit/VOC2012/SegmentationClass/2008_001896.png
VOCdevkit/VOC2012/SegmentationClass/2008_001926.png
VOCdevkit/VOC2012/SegmentationClass/2008_001966.png
VOCdevkit/VOC2012/SegmentationClass/2008_001971.png
VOCdevkit/VOC2012/SegmentationClass/2008_001992.png
VOCdevkit/VOC2012/SegmentationClass/2008_001997.png
VOCdevkit/VOC2012/SegmentationClass/2008_002032.png
VOCdevkit/VOC2012/SegmentationClass/2008_002043.png
VOCdevkit/VOC2012/SegmentationClass/2008_002064.png
VOCdevkit/VOC2012/SegmentationClass/2008_002066.png
VOCdevkit/VOC2012/SegmentationClass/2008_002067.png
VOCdevkit/VOC2012/SegmentationClass/2008_002073.png
VOCdevkit/VOC2012/SegmentationClass/2008_002079.png
VOCdevkit/VOC

In [3]:
cat /datasets/VOCdevkit/VOC2012/Annotations/2011_002897.xml

<annotation>
	<filename>2011_002897.jpg</filename>
	<folder>VOC2012</folder>
	<object>
		<name>bottle</name>
		<bndbox>
			<xmax>500</xmax>
			<xmin>473</xmin>
			<ymax>284</ymax>
			<ymin>202</ymin>
		</bndbox>
		<difficult>0</difficult>
		<occluded>0</occluded>
		<pose>Unspecified</pose>
		<truncated>1</truncated>
	</object>
	<segmented>0</segmented>
	<size>
		<depth>3</depth>
		<height>375</height>
		<width>500</width>
	</size>
	<source>
		<annotation>PASCAL VOC2011</annotation>
		<database>The VOC2011 Database</database>
		<image>flickr</image>
	</source>
</annotation>


In [5]:
import tensorflow as tf

In [7]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Input

def model_picker(name):
    if name == 'resnet':
        base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))

        input = Input(shape=(224,224,3))
        x = base_model(input)
        x = GlobalAveragePooling2D()(x)

        model = Model(inputs=input, outputs=x)
        return model

In [8]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.resnet50 import preprocess_input

root_dir = '/datasets/VOCdevkit/VOC2012'

datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

generator = datagen.flow_from_directory(
    root_dir,
    target_size=(224, 224),
    class_mode=None,
    shuffle=False,
    batch_size=64,
    classes=["JPEGImages"]
)

model_architecture = 'resnet'
model = model_picker(model_architecture)

voc_feature_list = model.predict(generator, verbose=1)

Found 17125 images belonging to 1 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
268/268 ━━━━━━━━━━━━━━━━━━━━ 3186s 12s/step


In [9]:
root_dir = '/datasets/VOCdevkit/VOC2012'

datagen = tensorflow.keras.preprocessing.image.ImageDataGenerator(preprocessing_function=preprocess_input)

generator = datagen.flow_from_directory(root_dir,
                                        target_size=(224, 224),
                                        class_mode=None,
                                        shuffle=False,
                                        batch_size=64,
                                        classes=["JPEGImages"]) # not really a class, just a hack to filter out unnecessary folders without creating class-based subfolders

model_architecture = 'resnet'  # Resnet has largest fraction of non-zero values compared to other keras imagenet pretrained models: https://youtu.be/-5BAepEE9I8?t=524
model = model_picker(model_architecture)

voc_feature_list = model.predict(generator, verbose=1)


NameError: name 'tensorflow' is not defined

In [10]:
voc_feature_list.shape

(17125, 2048)

## Checking pre-trained resnet works well on VOC2012

In [11]:
vocplane1 = extract_features('/datasets/VOCdevkit/VOC2012/JPEGImages/2007_000032.jpg', model)
vocplane2 = extract_features('/datasets/VOCdevkit/VOC2012/JPEGImages/2007_000033.jpg', model)
voccomputer1 = extract_features('/datasets/VOCdevkit/VOC2012/JPEGImages/2007_000039.jpg', model)

compare_metrics(vocplane1,vocplane2,voccomputer1)

NameError: name 'extract_features' is not defined

In [13]:
import numpy as np
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import preprocess_input

def extract_features(img_path, model):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    features = model.predict(img_array)
    return features.flatten()

In [14]:
vocplane1 = extract_features('/datasets/VOCdevkit/VOC2012/JPEGImages/2007_000032.jpg', model)
vocplane2 = extract_features('/datasets/VOCdevkit/VOC2012/JPEGImages/2007_000033.jpg', model)
voccomputer1 = extract_features('/datasets/VOCdevkit/VOC2012/JPEGImages/2007_000039.jpg', model)

compare_metrics(vocplane1, vocplane2, voccomputer1)

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 212ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 217ms/step


NameError: name 'compare_metrics' is not defined

In [15]:
import numpy as np
from numpy.linalg import norm

def compare_metrics(vec1, vec2, vec3):

    def cosine_similarity(a, b):
        return np.dot(a, b) / (norm(a) * norm(b))

    print("Euclidean: ", np.linalg.norm(vec1 - vec2), np.linalg.norm(vec1 - vec3))
    print("Dot product:", np.dot(vec1, vec2), np.dot(vec1, vec3))
    print("Cosine Similarity:", cosine_similarity(vec1, vec2), cosine_similarity(vec1, vec3))

In [16]:
compare_metrics(vocplane1, vocplane2, voccomputer1)

Euclidean:  39.091106 44.102104
Dot product: 1060.0789 756.1095
Cosine Similarity: 0.59598 0.45694715


In [18]:
import os

base_path = '/datasets/VOCdevkit/VOC2012/JPEGImages'

voc_filenames = [
    os.path.join(base_path, filepath)
    for filepath in sorted(os.listdir(base_path))
]

In [19]:
voc_filenames = ['/datasets/VOCdevkit/VOC2012' + filepath for filepath in sorted(os.listdir('/datasets/VOCdevkit/VOC2012/JPEGImages'))]

In [20]:
pickle.dump(voc_filenames, open('/features/filenames-voc2012.pickle', 'wb'))
pickle.dump(voc_feature_list,open('/features/features-voc2012-' + model_architecture + '.pickle', 'wb'))

NameError: name 'pickle' is not defined

In [21]:
import os
import pickle

# create folder properly
os.makedirs('/content/features', exist_ok=True)

# save files
pickle.dump(voc_filenames, open('/content/features/filenames-voc2012.pickle', 'wb'))

pickle.dump(
    voc_feature_list,
    open('/content/features/features-voc2012-' + model_architecture + '.pickle', 'wb')
)

In [22]:
FEATURE_PATH = "/content/drive/MyDrive/ahrefs/features"
os.makedirs(FEATURE_PATH, exist_ok=True)

pickle.dump(voc_filenames, open(FEATURE_PATH + '/filenames-voc2012.pickle', 'wb'))

pickle.dump(
    voc_feature_list,
    open(FEATURE_PATH + '/features-voc2012-' + model_architecture + '.pickle', 'wb')
)

In [23]:
# whole folder will be overwritten, may be better to cp individual files
!cp -r /features /content/gdrive/MyDrive/ahrefs

cp: cannot stat '/features': No such file or directory


In [24]:
!cp -r /content/features /content/drive/MyDrive/ahrefs/